# HSLS:09 Dropout Prediction — Modelling Functions

This notebook defines all reusable functions used across the modelling pipeline.
It is imported by the tier-specific modelling notebooks rather than run directly.
All functions assume the clean dataset (hsls_clean.csv), the train/test split
(train_test_split.json), and the feature tier lists (feature_tiers.json) have
already been prepared by the earlier notebooks.

In [25]:
import pandas as pd
import numpy as np
import json
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

In [ ]:
def fit_models(features, df, train_idx, test_idx):
    
    train = df.loc[train_idx]
    test  = df.loc[test_idx]
    
    features = [f for f in features if f in df.columns]
    
    X_train = train[features]
    y_train = train['X4EVERDROP']
    X_test  = test[features]
    y_test  = test['X4EVERDROP']
    
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    xgb = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    
    param_grid = {
        'n_estimators': [100, 200, 300, 500, 750, 1000],
        'max_depth': [3, 4, 5, 6, 7, 8, 10, 15],
        'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    }
    
    search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_grid,
        n_iter=100,
        scoring='f1',
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    search.fit(X_train, y_train)
    
    return search, X_train, y_train, X_test, y_test

In [29]:
def define_fine_grid(cv_results, top_k=3, range_pct=0.2):
    param_cols = ['param_n_estimators', 'param_max_depth', 'param_learning_rate',
                  'param_subsample', 'param_colsample_bytree']
    
    results = pd.DataFrame(cv_results)
    
    # Filter to competitive region — top range_pct of the score range
    max_score = results['mean_test_score'].max()
    min_score = results['mean_test_score'].min()
    threshold = max_score - (max_score - min_score) * range_pct
    competitive = results[results['mean_test_score'] >= threshold]
    
    top_n_results = competitive.sort_values('mean_test_score', ascending=False)[
        ['mean_test_score', 'std_test_score'] + param_cols
    ]
    
    print(f"Score range: {min_score:.4f} — {max_score:.4f}")
    print(f"Threshold: {threshold:.4f} ({len(top_n_results)} combinations above threshold)")
    print()
    print(top_n_results.to_string())
    print()
    
    for col in param_cols:
        print(top_n_results[col].value_counts())
        print()
    
    fine_grid = {}
    
    for col in param_cols:
        clean_col = col.replace('param_', '')
        counts = top_n_results[col].value_counts().sort_values(ascending=False)
        all_values = sorted(results[col].unique())
        
        # Take top k values by count, including ties at the boundary
        kth_count = counts.iloc[min(top_k, len(counts)) - 1]
        selected = sorted(counts[counts >= kth_count].index.tolist())
        
        # Fill gaps between selected values using values from the original grid
        filled = list(selected)
        for i in range(len(selected) - 1):
            lo = selected[i]
            hi = selected[i + 1]
            between = [v for v in all_values if lo < v < hi]
            filled.extend(between)
        filled = sorted(set(filled))
        
        fine_grid[clean_col] = filled
        print(f"{clean_col}: {filled}")
    
    n_combinations = 1
    for values in fine_grid.values():
        n_combinations *= len(values)
    
    print(f"\nTotal combinations: {n_combinations}")
    print(f"Total fits (x5 folds): {n_combinations * 5}")
    
    return fine_grid

In [ ]:
def fit_fine_search(fine_grid, X_train, y_train):
    
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    xgb = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    
    grid_search = GridSearchCV(
        estimator=xgb,
        param_grid=fine_grid,
        scoring='f1',
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    results = pd.DataFrame(grid_search.cv_results_)
    param_cols = [col for col in results.columns if col.startswith('param_')]
    top10 = results.sort_values('mean_test_score', ascending=False)[
        ['mean_test_score', 'std_test_score'] + param_cols
    ].head(10)
    
    print(top10.to_string())
    
    return grid_search

In [24]:
def evaluate_model(fine_search, X_train, y_train, X_test, y_test, tier):
    
    best_model = fine_search.best_estimator_
    
    y_pred_train = best_model.predict(X_train)
    y_prob_train = best_model.predict_proba(X_train)[:, 1]
    
    print(f"Tier {tier} — Training Set Evaluation")
    print(f"Accuracy:  {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_train, y_prob_train):.4f}")
    print()
    print(classification_report(y_train, y_pred_train))
    
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    
    print(f"Tier {tier} — Test Set Evaluation")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
    print()
    print(classification_report(y_test, y_pred))